# درس 10 - عوامل هوش مصنوعی در تولید

در این درس شما **الگوهای تولید** برای عوامل هوش مصنوعی را با استفاده از Microsoft Agent Framework (Python) خواهید آموخت. ما موضوعات زیر را پوشش می‌دهیم:

- **قابلیت مشاهده** — افزودن اندازه‌گیری زمان و ثبت وقایع به تعاملات عامل
- **ارزیابی** — استفاده از یک عامل ارزیاب برای امتیازدهی به کیفیت پاسخ
- **مدیریت هزینه** — راهکارهایی برای بهینه‌سازی توکن و انتخاب مدل

سناریو یک **عامل مسافرتی** است که به کاربران در برنامه‌ریزی سفرها کمک می‌کند، با نظارت و ارزیابی که بر آن افزوده شده‌اند.


## راه‌اندازی


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ملاحظات تولید

انتقال عامل‌های هوش مصنوعی از نمونه‌های اولیه به تولید نیازمند توجه دقیق به سه ستون:

1. **قابلیت مشاهده** — باید دید روشنی نسبت به کاری که عامل انجام می‌دهد، مدت زمانی که طول می‌کشد، و ابزارهایی که فراخوانی می‌کند داشته باشید. بدون ردیابی و لاگ‌گیری، اشکال‌زدایی مشکلات در محیط تولید تقریباً غیرممکن است.

2. **ارزیابی** — بررسی‌های خودکار کیفیت تضمین می‌کنند که پاسخ‌های عامل در طول زمان دقیق، کامل و مفید باقی بمانند. یک عامل ارزیاب می‌تواند پاسخ‌ها را بر اساس معیارهای تعریف‌شده امتیازدهی کند.

3. **مدیریت هزینه** — استفاده از توکن مستقیماً بر هزینه تأثیر می‌گذارد. راهکارهایی مانند بهینه‌سازی پرامپت، انتخاب مدل و استفاده از کش به کنترل هزینه‌ها بدون قربانی کردن کیفیت کمک می‌کنند.


## ساخت یک عامل قابل‌مشاهده

ما ابزارهای سفر را تعریف می‌کنیم و فراخوانی عامل را با زمان‌سنجی می‌پوشانیم تا بتوانیم تأخیر را پایش کنیم. در محیط تولید، شما آن را با OpenTelemetry یا یک بک‌اند ردیابی مشابه یکپارچه می‌کردید.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## الگوهای ارزیابی

یک الگوی رایج در تولید این است که از یک عامل دوم به‌عنوان **ارزیاب** استفاده شود. ارزیاب پاسخ عامل اصلی را بر اساس معیارهای از پیش تعریف‌شده‌ای مانند کامل بودن، دقت و مفید بودن امتیازدهی می‌کند.

این امکان را فراهم می‌کند:
- کنترل‌های کیفیت خودکار قبل از رسیدن پاسخ‌ها به کاربران
- شناسایی رگرسیون وقتی که پرامپت‌ها یا مدل‌ها تغییر می‌کنند
- پایش مداوم عملکرد عامل در طول زمان


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## استراتژی‌های مدیریت هزینه

کنترل هزینه‌ها برای عاملان هوش مصنوعی در محیط تولید حیاتی است. در اینجا استراتژی‌های کلیدی آمده‌اند:

| استراتژی | توضیحات |
|---|---|
| **بهینه‌سازی پرامپت** | دستورالعمل‌های سیستم را مختصر نگه دارید. زمینه‌های اضافی را حذف کنید تا تعداد توکن‌های ورودی کاهش یابد. |
| **انتخاب مدل** | از مدل‌های کوچکتر و ارزان‌تر (مثلاً GPT-4o-mini) برای کارهای ساده‌ای مانند طبقه‌بندی یا استخراج استفاده کنید، و مدل‌های بزرگ‌تر را برای استدلال پیچیده‌تر محفوظ نگه دارید. |
| **حافظهٔ نهان** | نتایج ابزارها و پرس‌وجوهای مکرر را در حافظهٔ نهان ذخیره کنید تا از فراخوانی‌های تکراری API جلوگیری شود. |
| **بودجهٔ توکن‌ها** | محدودیت `max_tokens` را تنظیم کنید تا از پاسخ‌های طولانی غیرمنتظره جلوگیری شود. |
| **تجمیع** | در صورت امکان، چندین پرسش کاربر را در یک فراخوانی API گروه‌بندی کنید. |

در عمل، یک رویکرد چندسطحی به‌خوبی عمل می‌کند: درخواست‌های ساده را به یک مدل سریع و کم‌هزینه هدایت کنید و تنها پرسش‌های پیچیده را به مدل قادرتر ارجاع دهید.


## خلاصه

در این درس یاد گرفتید چگونه:

1. **افزودن قابلیت مشاهده‌پذیری** به تعاملات عامل با زمان‌بندی و لاگ‌گیری، که زمینه را برای ردیابی و پایش فراهم می‌کند.
2. **ارزیابی پاسخ‌های عامل** به‌صورت خودکار با استفاده از یک عامل ارزیاب که کامل بودن، دقت و مفید بودن را امتیازدهی می‌کند.
3. **مدیریت هزینه‌ها** از طریق بهینه‌سازی پرامپت، انتخاب مدل، کش و بودجه توکن‌ها.

این الگوهای تولیدی به اطمینان از قابل‌اعتماد، قابل‌سنجش و مقرون‌به‌صرفه بودن عامل‌های هوش مصنوعی شما در مقیاس کمک می‌کنند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
سلب مسئولیت:
این سند با استفاده از سرویس ترجمهٔ هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی اشتباهات یا نادرستی‌هایی باشند. سند اصلی به زبان بومی خود باید به‌عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمهٔ حرفه‌ای توسط یک مترجم انسانی توصیه می‌شود. ما در قبال هرگونه سوءتفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
